# M5 — QADoomEnv tests and throughput benchmark on Kaggle

Settings: **Accelerator: GPU T4 x2**, **Internet: on**. Run with *Save & Run All* (commit mode).

Nothing here trains or touches the GPU. It runs on a GPU session anyway because throughput has to be measured on the CPUs that training will use (PLAN: Kaggle operating rule 8). Expect about 10 minutes.

1. The M5 tests, on Kaggle's Python 3.13 (the local venv is 3.12).
2. `scripts/bench_env.py`: steps/s for 1, 2, 4 and 16 workers on MAP07, the largest map. M5 needs **> 400 steps/s with 1 worker**.

Paste the output of the last two cells back to Claude.

In [ ]:
import os
# Must be set before anything imports torch or vizdoom (PLAN: Kaggle operating rules 6).
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

REPO_URL = "https://github.com/Satyajit-147/VizDoom.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/rl-qa-tester"

In [ ]:
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repo, or no secret attached

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e ".[dev]"

In [ ]:
import sys, platform, vizdoom as vzd, gymnasium
print("python   ", sys.version.split()[0], platform.platform())
print("vizdoom  ", vzd.__version__, "| gymnasium", gymnasium.__version__)
print("cpus     ", os.cpu_count())
!lscpu | grep -E "Model name|MHz" | head -3

In [ ]:
!python -m pytest tests/test_env_units.py tests/test_env_contract.py tests/test_reward_hacking.py tests/test_no_groundtruth_leak.py -q

In [ ]:
!python scripts/bench_env.py --workers 1 2 4 16 --seconds 20 --out /kaggle/working/bench_env.json